# 🚀 Colab T4 llama.cpp CUDA Server
Notebook ini menjalankan llama.cpp server dengan CUDA 12.8 backend, prebuilt latest otomatis diambil dari [`ai-dock/llama.cpp-cuda`](https://github.com/ai-dock/llama.cpp-cuda).  
Pilih model GGUF dari pilihan yang ada atau custom sendiri.  
Selanjutnya ekspos OpenAI-compatible API lewat Cloudflare Quick Tunnel.  
Satu notebook bisa dijalankan di Google Colab dan Kaggle.  
Tidak perlu menggunakan LiteLLM karena llama.cpp secara default sudah ekspos OpenAI-compatible API.

### Bagian
1. [Pengaturan](#pengaturan)
2. [Install LlamaCPP](#install-llamacpp)
3. [Download Model](#download-model)
4. [Jalankan Server](#jalankan-server)

# Pengaturan

### Preset Model
- [**Qwen3.6 35B Mini**](https://huggingface.co/SC117/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-GGUF) — Model MoE Qwen 3.6 multimodal (Image) & native MTP, APEX I-Mini
- [**Qwen3.6 35B Compact**](https://huggingface.co/SC117/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-GGUF) — Model MoE Qwen 3.6 multimodal (Image) & native MTP, APEX I-Compact
- [**Gemma 4 12B QAT**](https://huggingface.co/huihui-ai/Huihui-gemma-4-12B-it-qat-q4_0-unquantized-abliterated-GGUF) — Model Gemma 4 multimodal (Image, Audio, Video), MTP terpisah, Q4 QAT
- **Custom** — Model custom, silahkan atur sendiri

> Catatan: Semua model di preset adalah model yang tidak disensor / abliterated

### Rekomendasi Model
- Google Collab:  
  - Butuh speed: Qwen3.6 35B Mini dengan 16k konteks.  
  - Butuh konteks besar: Gemma 4 12B dengan 262k konteks.  
- Kaggle:  
  - Udah lah pakai Qwen3.6 35B Compact ajah bisa sampai 262k konteks, ngebut & konteks besar pula.  

### Penting Untuk Kaggle
Ubah pengaturan WORK_ROOT dan MODELS_ROOT.  
Secara default Kaggle directory: "/kaggle/working"  
Contoh:  
WORK_ROOT = "/kaggle/working/llamacpp"  
MODELS_ROOT = "/kaggle/working/models"  

In [ ]:
# @title 1. Pengaturan

# @markdown ### Model
MODEL_CHOICE = "qwen3.6-35b-mini"  # @param ["gemma-4-12b-qat", "qwen3.6-35b-mini", "qwen3.6-35b-compact", "Custom"]
MODEL_ALIAS = "character1"  # @param {type:"string"}

# @markdown ### Custom
# @markdown > Catatan: Hanya berlaku jika memilih model "Custom"
CUSTOM_NAME = "Model Lain"  # @param {type:"string"}
CUSTOM_ALIAS = "character1"  # @param {type:"string"}
CUSTOM_MODEL_URL = ""  # @param {type:"string"}
CUSTOM_MMPROJ_URL = ""  # @param {type:"string"}
CUSTOM_DRAFT_URL = ""  # @param {type:"string"}
CUSTOM_IS_MOE = False  # @param {type:"boolean"}
CUSTOM_HAS_NATIVE_MTP = False  # @param {type:"boolean"}
DOWNLOAD_MMPROJ = True  # @param {type:"boolean"}
DOWNLOAD_DRAFT_MODEL = True  # @param {type:"boolean"}

# @markdown ### Downloader
HF_TOKEN = ""  # @param {type:"string"}
HF_XET_HIGH_PERFORMANCE = True  # @param {type:"boolean"}
ARIA_CONNECTIONS = 16  # @param {type:"integer"}

# @markdown ### llama.cpp prebuilt release
LLAMA_CPP_RELEASE = "latest"  # @param {type:"string"}
FALLBACK_RELEASE_TAG = "b10250" # @param {type:"string"}

# @markdown ### Lain-lain
WORK_ROOT = "/content/llamacpp" # @param {type:"string"}
MODELS_ROOT = "/content/models" # @param {type:"string"}

# Install LlamaCPP

In [ ]:
# @title 2. Install llama.cpp
import os, re, sys, json, time, shutil, tarfile, subprocess
from pathlib import Path

import requests

REPO = "ai-dock/llama.cpp-cuda"
work_root = Path(WORK_ROOT)
archive_path = work_root / "llama.cpp-cuda.tar.gz"
extract_root = work_root / "runtime"
work_root.mkdir(parents=True, exist_ok=True)

print("Menginstall download helpers...", flush=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"],
    check=True,
)

def release_info(requested):
    if requested.strip().lower() == "latest":
        api_url = f"https://api.github.com/repos/{REPO}/releases/latest"
    else:
        api_url = f"https://api.github.com/repos/{REPO}/releases/tags/{requested.strip()}"

    response = requests.get(api_url, timeout=30)
    if response.ok:
        data = response.json()
        tag = data["tag_name"]
        candidates = [
            asset for asset in data.get("assets", [])
            if asset.get("name", "").endswith("cuda-12.8-amd64.tar.gz")
        ]
        if candidates:
            return tag, candidates[0]["browser_download_url"], candidates[0]["name"]
        expected = f"llama.cpp-{tag}-cuda-12.8-amd64.tar.gz"
        return tag, f"https://github.com/{REPO}/releases/download/{tag}/{expected}", expected

    if requested.strip().lower() == "latest":
        redirect = requests.get(
            f"https://github.com/{REPO}/releases/latest",
            allow_redirects=True,
            timeout=30,
        )
        match = re.search(r"/releases/tag/([^/?#]+)", redirect.url)
        tag = match.group(1) if match else FALLBACK_RELEASE_TAG
    else:
        tag = requested.strip()
    name = f"llama.cpp-{tag}-cuda-12.8-amd64.tar.gz"
    return tag, f"https://github.com/{REPO}/releases/download/{tag}/{name}", name

tag, release_url, asset_name = release_info(LLAMA_CPP_RELEASE)
print(f"Release: {tag}")
print(f"Asset  : {asset_name}")

curl_download = subprocess.run([
    "curl", "--fail", "--location", "--retry", "5", "--retry-all-errors",
    "--continue-at", "-", "--progress-bar",
    "--output", str(archive_path), release_url,
])
if curl_download.returncode != 0:
    raise RuntimeError(
        f"Gagal download: {release_url}"
    )

if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with tarfile.open(archive_path, "r:gz") as archive:
    archive.extractall(extract_root, filter="data")

servers = sorted(
    (p for p in extract_root.rglob("llama-server") if p.is_file()),
    key=lambda p: (len(p.parts), str(p)),
)
if not servers:
    raise FileNotFoundError("llama-server tidak ditemukan")

LLAMA_SERVER_PATH = servers[0].resolve()
LLAMA_CPP_DIR = LLAMA_SERVER_PATH.parent
LLAMA_SERVER_PATH.chmod(LLAMA_SERVER_PATH.stat().st_mode | 0o111)

print(f"llama-server: {LLAMA_SERVER_PATH}")
version = subprocess.run(
    [str(LLAMA_SERVER_PATH), "--version"],
    cwd=LLAMA_CPP_DIR,
    text=True,
    capture_output=True,
)
print((version.stdout or version.stderr).strip())
if version.returncode != 0:
    raise RuntimeError("llama-server gagal dimuat")

# Download Model

In [ ]:
# @title 3. Model
from pathlib import Path
from urllib.parse import unquote, urlparse
import os, re, subprocess

HF_TOKEN = HF_TOKEN.strip()

if HF_XET_HIGH_PERFORMANCE:
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
else:
    os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

from huggingface_hub import hf_hub_download

PRESETS = {
    "qwen3.6-35b-mini": {
        "alias": "character1",
        "model_url": "https://huggingface.co/SC117/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-GGUF/resolve/main/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-I-MINI.gguf?download=true",
        "mmproj_url": "https://huggingface.co/SC117/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-GGUF/resolve/main/mmproj-Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-F16.gguf?download=true",
        "draft_url": "",
        "is_moe": True,
        "native_mtp": True,
    },
    "qwen3.6-35b-compact": {
        "alias": "character1",
        "model_url": "https://huggingface.co/SC117/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-GGUF/resolve/main/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-I-Compact.gguf?download=true",
        "mmproj_url": "https://huggingface.co/SC117/Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-GGUF/resolve/main/mmproj-Qwen3.6-35B-A3B-uncensored-heretic-Native-MTP-Preserved-APEX-F16.gguf?download=true",
        "draft_url": "",
        "is_moe": True,
        "native_mtp": True,
    },
    "gemma-4-12b-qat": {
        "alias": "character1",
        "model_url": "https://huggingface.co/huihui-ai/Huihui-gemma-4-12B-it-qat-q4_0-unquantized-abliterated-GGUF/resolve/main/Huihui-gemma-4-12B-it-qat-q4_0-unquantized-abliterated-Q4_K.gguf?download=true",
        "mmproj_url": "https://huggingface.co/huihui-ai/Huihui-gemma-4-12B-it-qat-q4_0-unquantized-abliterated-GGUF/resolve/main/mmproj-model-bf16.gguf?download=true",
        "draft_url": "https://huggingface.co/huihui-ai/Huihui-gemma-4-12B-it-qat-q4_0-unquantized-abliterated-GGUF/resolve/main/mtp-ggml-model-bf16.gguf?download=true",
        "is_moe": False,
        "native_mtp": False,
    },
}

if MODEL_CHOICE == "Custom":
    selected = {
        "alias": CUSTOM_ALIAS.strip() or "character1",
        "model_url": CUSTOM_MODEL_URL.strip(),
        "mmproj_url": CUSTOM_MMPROJ_URL.strip(),
        "draft_url": CUSTOM_DRAFT_URL.strip(),
        "is_moe": CUSTOM_IS_MOE,
        "native_mtp": CUSTOM_HAS_NATIVE_MTP,
    }
    selected_name = CUSTOM_NAME.strip() or "Model Lain"
else:
    if MODEL_CHOICE not in PRESETS:
        raise ValueError(f"Model tidak valid! MODEL_CHOICE: {MODEL_CHOICE}")
    selected = PRESETS[MODEL_CHOICE].copy()
    selected_name = MODEL_CHOICE

if not selected["model_url"]:
    raise ValueError("URL model dibutuhkan")

MODEL_IS_MOE = bool(selected["is_moe"])
MODEL_HAS_NATIVE_MTP = bool(selected["native_mtp"])
RESOLVED_ALIAS = MODEL_ALIAS.strip() or selected["alias"]
safe_dir = re.sub(r"[^a-zA-Z0-9._-]+", "-", RESOLVED_ALIAS).strip("-") or "model"
model_dir = Path(MODELS_ROOT) / safe_dir
model_dir.mkdir(parents=True, exist_ok=True)

MODEL_PATH = model_dir / "model.gguf"
MMPROJ_PATH = model_dir / "mmproj.gguf"
DRAFT_MODEL_PATH = model_dir / "draft.gguf"

def fetch(label, url, destination):
    if not url:
        return None
    parsed = urlparse(url)
    parts = [unquote(part) for part in parsed.path.split("/") if part]
    is_hf_resolve = (
        parsed.netloc.lower() in {"huggingface.co", "www.huggingface.co"}
        and len(parts) >= 5
        and parts[2] == "resolve"
    )

    if is_hf_resolve:
        repo_id = "/".join(parts[:2])
        revision = parts[3]
        filename = "/".join(parts[4:])
        print(
            f"\nDownloading {label} with Hugging Face Xet\n"
            f"  repo: {repo_id}\n  file: {filename}",
            flush=True,
        )
        download_args = dict(
            repo_id=repo_id,
            filename=filename,
            revision=revision,
            cache_dir=str(Path(WORK_ROOT) / "hf-cache"),
        )
        if HF_TOKEN:
            download_args["token"] = HF_TOKEN
        downloaded = Path(hf_hub_download(**download_args)).resolve()
        if destination.exists() or destination.is_symlink():
            destination.unlink()
        destination.symlink_to(downloaded)
    else:
        existing = destination.stat().st_size if destination.exists() else 0
        action = f"meneruskan dari {existing / 2**30:.2f} GiB" if existing else "memulai"
        print(f"\nMendownload {label} dengan aria2c ({action}) → {destination}", flush=True)
        connections = max(1, min(int(ARIA_CONNECTIONS), 16))
        result = subprocess.run([
            "aria2c", "--continue=true",
            f"--max-connection-per-server={connections}", f"--split={connections}",
            "--min-split-size=4M", "--file-allocation=none", "--max-tries=0",
            "--retry-wait=3", "--max-redirect=20", "--summary-interval=1",
            "--auto-file-renaming=false", "--allow-overwrite=true",
            "--dir", str(destination.parent), "--out", destination.name, url,
        ])
        if result.returncode != 0:
            raise RuntimeError(f"Gagal download {label}")

    print(f"Selesai {label}: {destination.stat().st_size / 2**30:.2f} GiB", flush=True)
    return destination

fetch("model", selected["model_url"], MODEL_PATH)
RESOLVED_MMPROJ_PATH = (
    fetch("mmproj", selected["mmproj_url"], MMPROJ_PATH)
    if DOWNLOAD_MMPROJ and selected["mmproj_url"] else None
)
RESOLVED_DRAFT_PATH = (
    fetch("draft model", selected["draft_url"], DRAFT_MODEL_PATH)
    if DOWNLOAD_DRAFT_MODEL and selected["draft_url"] else None
)

print("\nSiap:")
print("  Nama      :", selected_name)
print("  API alias :", RESOLVED_ALIAS)
print("  Model     :", MODEL_PATH)
print("  mmproj    :", RESOLVED_MMPROJ_PATH or "disabled")
print("  Draft     :", RESOLVED_DRAFT_PATH or ("native MTP" if MODEL_HAS_NATIVE_MTP else "disabled"))
print("  MoE       :", MODEL_IS_MOE)

# Jalankan Server

### Rekomendasi Pengaturan
- Google Colab T4:
  - Qwen3.6 35B Mini (Cepat - Konteks kecil):
    - CONTEXT_SIZE: 16384
    - N_CPU_MOE: 4
    - BATCH_SIZE: 2048
    - UBATCH_SIZE: 512
  - Qwen3.6 35B Mini (Lambat - Konteks besar):
    - CONTEXT_SIZE: 262144
    - N_CPU_MOE: 19
    - BATCH_SIZE: 2048
    - UBATCH_SIZE: 512
  - Gemma 4 12B QAT:
    - CONTEXT_SIZE: 262144
    - BATCH_SIZE: 4096
    - UBATCH_SIZE: 2048
- Kaggle 2x T4:
  - Qwen3.6 35B Compact (Cepat - Konteks Besar):
    - CONTEXT_SIZE: 262144
    - N_CPU_MOE: 0
    - BATCH_SIZE: 2048
    - UBATCH_SIZE: 1024

> Catatan: Untuk nilai yang lain biarkan default saja, atau silahkan bereksperimen juga boleh

In [ ]:
# @title 4. Jalankan llama-server dan Cloudflare tunnel

# @markdown ### Runtime
ENABLE_THINKING = False # @param {type:"boolean"}
CONTEXT_SIZE = 16384  # @param {type:"integer"}
CACHE_TYPE_K = "q8_0"  # @param ["f16", "q8_0", "q4_0", "q4_1", "iq4_nl", "q5_0", "q5_1"]
CACHE_TYPE_V = "q8_0"  # @param ["f16", "q8_0", "q4_0", "q4_1", "iq4_nl", "q5_0", "q5_1"]
GPU_LAYERS = "99"  # @param {type:"string"}
BATCH_SIZE = 2048  # @param {type:"integer"}
UBATCH_SIZE = 512  # @param {type:"integer"}
PARALLEL_SLOTS = 1  # @param {type:"integer"}
FLASH_ATTN = "auto"  # @param ["auto", "on", "off"]
MMAP = False  # @param {type:"boolean"}
FIT_MODE = "off"  # @param ["on", "off"]
FIT_TARGET_MIB = 1024  # @param {type:"integer"}

# @markdown ### Speculative Decoding
SPEC_TYPE = "draft-mtp"  # @param ["none", "draft-mtp", "draft-simple", "draft-eagle3", "draft-dflash", "ngram-simple", "ngram-map-k", "ngram-map-k4v", "ngram-mod", "ngram-cache"]
SPEC_DRAFT_N_MAX = 3  # @param {type:"integer"}
DRAFT_CACHE_TYPE_K = "q8_0"  # @param ["f16", "q8_0", "q4_0", "q4_1", "iq4_nl", "q5_0", "q5_1"]
DRAFT_CACHE_TYPE_V = "q8_0"  # @param ["f16", "q8_0", "q4_0", "q4_1", "iq4_nl", "q5_0", "q5_1"]

# @markdown ### CPU MoE
# @markdown > Catatan:<br>
# @markdown > Hanya berlaku untuk model MoE<br>
# @markdown > Set 0 = semua expert masuk ke VRAM<br>
# @markdown > Angka Expert semakin besar = beban VRAM semakin ringan tapi jadi lambat<br>
# @markdown > Naikkan jika menggunakan model MoE dan terdapat error "out of memory" atau crash
N_CPU_MOE = 4  # @param {type:"integer"}

# @markdown ### Server
PORT = 12345  # @param {type:"integer"}
API_KEY = "sk-colab-local"  # @param {type:"string"}
ENABLE_WEBUI = True  # @param {type:"boolean"}
HOST = "0.0.0.0"    # @param {type:"string"}
EXTRA_SERVER_ARGS = ""  # @param {type:"string"}

import os, re, shlex, time, subprocess
from pathlib import Path
import requests

required = ["LLAMA_SERVER_PATH", "LLAMA_CPP_DIR", "MODEL_PATH", "RESOLVED_ALIAS"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Jalankan cells sebelumnya dulu. Gagal dimuat: {missing}")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    text=True, capture_output=True,
)
if gpu.returncode != 0:
    raise RuntimeError("Tidak ada NVIDIA GPU terdeteksi. Harap ubah runtime Colab menjadi GPU dahulu.")
print("GPU:", gpu.stdout.strip())

cmd = [
    str(LLAMA_SERVER_PATH),
    "-m", str(MODEL_PATH),
    "-a", RESOLVED_ALIAS,
    "-ctk", CACHE_TYPE_K,
    "-ctv", CACHE_TYPE_V,
    "-c", str(CONTEXT_SIZE),
    "-ngl", str(GPU_LAYERS),
    "-b", str(BATCH_SIZE),
    "-ub", str(UBATCH_SIZE),
    "-np", str(PARALLEL_SLOTS),
    "--flash-attn", FLASH_ATTN,
    "--fit", FIT_MODE,
    "--jinja",
    "--host", HOST,
    "--port", str(PORT),
]

cmd += ["--webui" if ENABLE_WEBUI else "--no-webui"]
if API_KEY:
    cmd += ["--api-key", API_KEY]
if RESOLVED_MMPROJ_PATH:
    cmd += ["--mmproj", str(RESOLVED_MMPROJ_PATH)]
if MODEL_IS_MOE and int(N_CPU_MOE) > 0:
    cmd += ["-ncmoe", str(N_CPU_MOE)]
if FIT_MODE == "on":
    cmd += ["--fit-target", str(FIT_TARGET_MIB)]
cmd += ["--reasoning", "on" if ENABLE_THINKING else "off"]
cmd += ["--mmap" if MMAP else "--no-mmap"]
if SPEC_TYPE != "none":
    cmd += [
        "--spec-type", SPEC_TYPE,
        "--spec-draft-n-max", str(SPEC_DRAFT_N_MAX),
        "--cache-type-k-draft", DRAFT_CACHE_TYPE_K,
        "--cache-type-v-draft", DRAFT_CACHE_TYPE_V,
    ]
    if RESOLVED_DRAFT_PATH:
        cmd += ["--model-draft", str(RESOLVED_DRAFT_PATH)]

if EXTRA_SERVER_ARGS.strip():
    cmd += shlex.split(EXTRA_SERVER_ARGS)

SERVER_LOG = "/tmp/llama-server.log"
print("\nMenjalankan:", LLAMA_CPP_DIR)
print("Command:", shlex.join(cmd))
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        cmd,
        cwd=LLAMA_CPP_DIR,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
models_url = f"http://127.0.0.1:{PORT}/v1/models"
started = time.time()
while time.time() - started < 1200:
    if server_proc.poll() is not None:
        tail = "".join(Path(SERVER_LOG).read_text(errors="replace").splitlines(keepends=True)[-120:])
        raise RuntimeError(f"llama-server crash. Log tail:\n{tail}")
    try:
        if requests.get(models_url, headers=headers, timeout=3).status_code == 200:
            break
    except requests.RequestException:
        pass
    elapsed = int(time.time() - started)
    if elapsed and elapsed % 30 == 0:
        print(f"Memuat model... {elapsed}s")
    time.sleep(1)
else:
    raise TimeoutError(f"llama-server timed out. Periksa {SERVER_LOG}")

print("llama-server siap")

cloudflared = Path(WORK_ROOT) / "cloudflared"
if not cloudflared.exists():
    result = subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(cloudflared),
    ])
    if result.returncode != 0:
        server_proc.terminate()
        raise RuntimeError("Gagal mendownload cloudflared")
    cloudflared.chmod(0o755)

CF_LOG = "/tmp/cloudflared.log"
with open(CF_LOG, "w") as log:
    tunnel_proc = subprocess.Popen(
        [str(cloudflared), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

tunnel_url = None
for _ in range(120):
    if tunnel_proc.poll() is not None:
        break
    text = Path(CF_LOG).read_text(errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if not tunnel_url:
    server_proc.terminate()
    tail = "".join(Path(CF_LOG).read_text(errors="replace").splitlines(keepends=True)[-80:])
    raise RuntimeError(f"Cloudflare tunnel URL tidak ditemukan. Log tail:\n{tail}")

BASE_URL = tunnel_url + "/v1"
print("\n" + "=" * 72)
print("SERVER SIAP")
if ENABLE_WEBUI:
    print("WEBUI =", tunnel_url)
print("BASE_URL =", BASE_URL)
print("API_KEY  =", API_KEY or "(none)")
print("MODEL    =", RESOLVED_ALIAS)
print("=" * 72)

payload = (
    '{"model":"' + RESOLVED_ALIAS + '",'
    '"messages":[{"role":"user","content":"Say hello in one sentence."}],'
    '"max_tokens":64}'
)
auth = f" -H 'Authorization: Bearer {API_KEY}'" if API_KEY else ""
print("\nSmoke test:")
print(f"curl {BASE_URL}/chat/completions{auth} -H 'Content-Type: application/json' -d '{payload}'")
print("\nBiarkan cell ini tetap running saat menggunakan local LLM. Stop/interrupt akan mematikan server.\n")

tick = 0
try:
    while True:
        time.sleep(60)
        tick += 1
        healthy = server_proc.poll() is None and tunnel_proc.poll() is None
        print(f"[{time.strftime('%H:%M:%S')}] heartbeat #{tick:04d} | " + ("healthy" if healthy else "process stopped"))
        if not healthy:
            break
except KeyboardInterrupt:
    print("\nMenghentikan server dan tunnel...")
finally:
    for proc in (tunnel_proc, server_proc):
        try:
            proc.terminate()
            proc.wait(timeout=10)
        except Exception:
            try:
                proc.kill()
            except Exception:
                pass
    print("Dihentikan.")